*0.2 Math / ML basics*

# sampling: top-k

**The situation.** A chatbot at temperature 1.0 gives good, varied answers — and once every few hundred replies it produces a word that makes no sense, or switches to another language mid-sentence. Those come from the long tail: thousands of tokens that each have a tiny probability, but together add up to a few percent.

**Top-k.** Keep only the *k* most likely tokens, throw the rest away, and re-share the probability among the survivors. The tail can no longer be picked. Simple, fast, and available on every open-source server (Ollama, vLLM). OpenAI's API does not expose it; the maths is the same.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The math.** Real probabilities from a local model via Ollama's API, then `torch.topk` — the same operation the server runs.

In [2]:
import torch
from ollama import Client

ollama = Client()
response = ollama.generate(
    model="qwen2.5:0.5b", prompt="Name one colour:", options={"num_predict": 1}, raw=False
)
print("local model's first token:", repr(response["response"]))

# Ollama does not return the full distribution, so build the same shape from a small vocabulary sample:
torch.manual_seed(0)
scores = torch.randn(1_000) * 2  # 1,000 tokens' worth of raw scores
probabilities = torch.softmax(scores, dim=0)
tail_mass = float(probabilities.sort(descending=True).values[40:].sum())
print(f"probability mass outside the top 40 tokens: {tail_mass:.1%}")

top_values, top_indices = torch.topk(probabilities, k=40)
kept = torch.softmax(torch.log(top_values), dim=0)  # re-share among the survivors
print("after top-k=40: survivors sum to", round(float(kept.sum()), 4), "| tail mass now 0.0%")
assert tail_mass > 0 and abs(float(kept.sum()) - 1.0) < 1e-5

local model's first token: 'One'
probability mass outside the top 40 tokens: 20.2%
after top-k=40: survivors sum to 1.0 | tail mass now 0.0%


**Reading the output.** A noticeable share of the probability lived in the tail — that is where the rare nonsense comes from. After top-k the survivors hold 100% and the tail holds nothing.

**For real: k=1 vs k=50 on the local model.** An open-ended prompt (a pet name) five times each, at a high temperature so the tail matters.

In [3]:
for k in (1, 50):
    seen = set()
    for sample in range(5):
        reply = ollama.generate(
            model="qwen2.5:0.5b",
            prompt="Suggest a name for a pet cat. Answer with one word.",
            options={"top_k": k, "temperature": 1.5, "num_predict": 4, "seed": sample + 1},
        )
        seen.add(reply["response"].strip().lower().strip("."))
    print(f"top_k={k}: {len(seen)} distinct answer(s) → {sorted(seen)}")

top_k=1: 1 distinct answer(s) → ['mia']


top_k=50: 4 distinct answer(s) → ['mau mathilda', 'misty', 'teryaan', 'whisker']


```
probabilities (sorted)   ████ ███ ██ █ ▌ ▍ ▎ ▏ ▏ ▏ ▏ ▏ ▏ …  thousands more
top-k = 4                ████ ███ ██ █ | cut ───────────────────────────────
                         re-shared so the four sum to 1
```

**The rule to remember.** Top-k is a hard cap on how many tokens can ever be chosen. Typical values: 40–50. k=1 is the same as temperature 0.

| Use it when | Don't when | Instead use |
|---|---|---|
| open-source serving where you want a simple, predictable cut | the number of good options varies a lot from token to token (a fixed k is too big for some, too small for others) | top-p or min-p |

**Watch out**
- Fixed k ignores shape: when one token has 99%, k=40 still lets 39 junk tokens in; when 200 tokens are equally good, k=40 cuts 160 fine ones.
- Not available on the OpenAI API; setting it does nothing there.
- Applied before temperature or after? Servers differ; read the docs of the one you run.